# Simulating a beta-binomial dataset for the duplex error model

Generates the synthetic dataset used by **Supplementary Fig. 11a,b** — `Simulation_v2_beta_binomials_40_samples_delta_initialsied_from_variance_or_MOM.csv`.

Each simulated "position" stands for one position in the SNV/indel panel, sequenced across 40 samples on
one lane. Variant read counts are drawn from a beta-binomial with a known error rate and dispersion, so the
true parameters are known exactly and the variant caller's false-discovery rate can be measured against them.

The parameter grid:

- position error rate `epsilon`, 50 values log-spaced from 1e-4 to 1
- dispersion `delta`, 50 values log-spaced from 1e-5 to 1e3
- 40 samples per position, all at a fixed depth of 1800 reads, chosen to match the observed DCS depth

Where `delta * epsilon < 1e-7` the beta-binomial is indistinguishable from its binomial limit and the counts
are drawn from a binomial instead.

The last value on the `epsilon` grid, 1.0, cannot be simulated: the beta-binomial's second shape parameter
`b = (1 - epsilon)/(delta * epsilon)` is zero there, which `scipy.stats.betabinom` rejects. That value is
skipped, so 49 x 50 = 2,450 combinations are simulated. A further 85 are dropped because all 40 samples drew
zero variant reads, leaving **2,365 distinct positions**.

For each position, `epsilon` and `delta` are then re-estimated from the simulated counts by maximum
likelihood, initialised two ways (`delta` from the sample variance, and `delta` from method of moments), so
that the accuracy of the fitting procedure can be assessed. Supplementary Fig. 11 itself uses the *true*
`actual_epsilon` and `actual_delta`, not these estimates.

## Data availability

Everything here is synthetic — no participant data is used or produced. The output is included in this
repository as `Data_files/Simulation_v2_beta_binomials_40_samples_delta_initialsied_from_variance_or_MOM.csv`.

The simulation is not seeded, so re-running produces a statistically equivalent but not byte-identical file.
To reproduce Supplementary Fig. 11a,b exactly, use the file already in `Data_files/`.

In [ ]:
# imported packages
import numpy as np
import pandas as pd
import scipy.optimize
import scipy.stats
from scipy.stats import betabinom
import time

## Beta-binomial likelihood and fitting

In [ ]:
def BB_likelihood(params, var_depths, total_depths):
    epsilon = params[0]
    delta = params[1]
    a = 1/delta
    b = (1-epsilon)/(delta*epsilon)

    log_sample_BBs = []

    for sample_variant_depth, sample_total_depth in zip(var_depths, total_depths):
        BB = betabinom.pmf(sample_variant_depth, sample_total_depth, a, b) #beta-binomial likelihood of that number of variant reads, given the sample depth, position error rate and delta
        log_sample_BBs.append(np.log(BB))

    model_likelihood_log = np.sum(log_sample_BBs)

    #reject parameter combinations outside the permitted range
    if delta<1e-08/epsilon:
        model_likelihood_log = -100000000000

    if epsilon<0:
        model_likelihood_log = -100000000000

    if epsilon>1:
        model_likelihood_log = -100000000000

    return -model_likelihood_log

In [ ]:
def a_b(epsilon, delta):
    a = 1/delta
    b = (1-epsilon)/(delta*epsilon)
    return a, b

In [ ]:
def estimate_delta_method_of_moments(variant_reads, total_depth_reads):
    number_samples = len(variant_reads)
    mean_depth = np.mean(total_depth_reads)

    #first raw sample moment
    k = 1
    m_1 = (sum([var_read**k for var_read in variant_reads]))/number_samples

    #second raw sample moment
    k = 2
    m_2 = (sum([var_read**k for var_read in variant_reads]))/number_samples

    alpha = (mean_depth*m_1-m_2)/(mean_depth*((m_2/m_1)-m_1-1)+m_1)

    delta = 1/alpha

    return delta

In [ ]:
def beta_binomial_MLE(variant_reads_list, sample_depths, position_error_rate_initial_guess, position_delta_initial_guess):

    initial_guess = [position_error_rate_initial_guess, position_delta_initial_guess]

    #calculate the most likely epsilon and delta
    optimization = scipy.optimize.minimize(BB_likelihood, initial_guess, args=(variant_reads_list, sample_depths, ),
                                            method='Nelder-Mead', options = {'maxiter': 10000, 'maxfev': 10000})
    optimization_minima = optimization['fun']
    optimization_outcome = optimization['success']
    optimization_message = optimization['message']
    epsilon = optimization['x'][0]
    delta = optimization['x'][1]

    position_MLE_results = (epsilon, delta, optimization_outcome, optimization_message, optimization_minima)

    return position_MLE_results

## Run the simulation

Takes a few hours: two Nelder-Mead optimisations per position, over 2,450 grid points.

The loop skips the final `epsilon` value of 1.0 for the reason given above. One row is written per
simulated position, giving 2,365 rows.

In [ ]:
#if delta*epsilon <1e-07, model as a binomial, otherwise beta-binomial
error_rates = np.logspace(-4, 0, 50) #logspaced error rates between 1e-4 and 1e0
deltas = np.logspace(-5, 3, 50) #logspaced deltas between 1e-5 and 1e3
n = 1800
samples = 40

sim_40 = []

positions = 0
start_time = time.time()
for epsilon in error_rates:
    if epsilon >= 1: #b = (1-epsilon)/(delta*epsilon) is 0 at epsilon = 1, which the beta-binomial does not accept
        continue
    for delta in deltas:
        total_depth_reads = [n]*samples
        if delta*epsilon>1e-07:
            a, b = a_b(epsilon, delta)
            variant_reads = list(scipy.stats.betabinom.rvs(n, a, b, size = samples))
        else: #binomial
            variant_reads = list(np.random.binomial(n, epsilon, samples)) #draw 40 samples from a binomial with depth n and error rate epsilon

        if np.sum(variant_reads)>0:
            epsilon_estimated = np.sum(variant_reads)/np.sum(total_depth_reads)

            #MLE using delta initialised from the variance
            mean_depth = np.mean(total_depth_reads)
            variance = np.var(variant_reads)
            var_delta_estimate = (variance-(mean_depth*epsilon_estimated))/(mean_depth*mean_depth*epsilon_estimated)

            if var_delta_estimate>0:
                var_MLE_outcome = beta_binomial_MLE(variant_reads, total_depth_reads, epsilon_estimated, var_delta_estimate)
            else:
                var_MLE_outcome = beta_binomial_MLE(variant_reads, total_depth_reads, epsilon_estimated, 1e-04/epsilon_estimated)

            #MLE using delta initialised from method of moments
            MOM_delta_estimate = estimate_delta_method_of_moments(variant_reads, total_depth_reads)

            if MOM_delta_estimate>0:
                MOM_MLE_outcome = beta_binomial_MLE(variant_reads, total_depth_reads, epsilon_estimated, MOM_delta_estimate)
            else:
                MOM_MLE_outcome = beta_binomial_MLE(variant_reads, total_depth_reads, epsilon_estimated, 1e-04/epsilon_estimated)

            sim_40.append((epsilon, delta, delta*epsilon, epsilon_estimated, var_delta_estimate, MOM_delta_estimate,
                           var_MLE_outcome, MOM_MLE_outcome, variant_reads, total_depth_reads))

        positions+=1
        if positions%10==0:
            print('positions done = ', positions)
            print('time for last 10 positions = %s seconds' % int(time.time() - start_time))
            start_time = time.time()

## Save

In [ ]:
simulated_positions_df_40 = pd.DataFrame(sim_40, columns =['actual_epsilon', 'actual_delta', 'delta*epsilon',
                                                           'epsilon_estimate', 'delta_estimate_from_variance',
                                                           'MOM_delta_estimate', 'MLE results (initialised variance delta)',
                                                           'MLE results (initialised MOM delta)', 'variant_reads', 'depth_reads'])

simulated_positions_df_40.to_csv('Data_files/Simulation_v2_beta_binomials_40_samples_delta_initialsied_from_variance_or_MOM.csv')

simulated_positions_df_40